# Brain-region classification: analysis and results

Set `BRAIN_REGION_DATA_FILE` in the first code cell if the input CSV is not beside this repository. Results live in `results/`; generated figures and tables can be inspected there.

The notebooks call functions in the four Python modules. Saved summaries are loaded when present; a fresh nested grid can take many hours.


In [ ]:
# Project-wide paths: change DATA_FILE here or set BRAIN_REGION_DATA_FILE.
import os
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
DATA_FILE = Path(os.environ.get("BRAIN_REGION_DATA_FILE",
                            PROJECT_DIR.parent / "GSE211792_logCPM_cent.csv")).expanduser().resolve()
RESULTS_DIR = PROJECT_DIR / "results"
RESUME_CHECKPOINT_DIR = PROJECT_DIR / "resume_checkpoints"

assert DATA_FILE.is_file(), "Set BRAIN_REGION_DATA_FILE to the input CSV before running."
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Input CSV found; result directory ready.")


## 1. Nested donor-held-out experiments

Runs direct 25-region, binary cell-type, 4- and 7-macro, and separate two-stage models. ANOVA, MI, and PCA are tested with linear and RBF SVMs. Hyperparameters and preprocessing are fitted within the training donors.

In [ ]:
from IPython.display import display
from project_analysis import (run_nested_models, run_pca_variance_search,
                              run_fixed_comparisons, save_comparison_tables)

nested_summary = run_nested_models(PROJECT_DIR, DATA_FILE, RESULTS_DIR,
                                   RESUME_CHECKPOINT_DIR)
display(nested_summary.head(12))
print(f"Saved {len(nested_summary)} nested-grid summaries")


## 2. Direct PCA variance-target search

Inner folds choose among 50%, 70%, 90%, 95%, and 99% variance targets for linear and RBF SVMs.

In [ ]:
pca_variance_summary = run_pca_variance_search(PROJECT_DIR, DATA_FILE, RESULTS_DIR)
display(pca_variance_summary)

## 3. Joint-loss and cell-specific models

The joint model, neuronal-only and non-neuronal-only fits, and the cross-family comparison are implemented in the two model modules. These fixed-setting runs are exploratory; they are marked separately from the nested-tuned runs in the saved tables.

In [ ]:
joint_summary = run_fixed_comparisons(PROJECT_DIR, DATA_FILE, RESULTS_DIR, RESUME_CHECKPOINT_DIR)
display(joint_summary)

## 4. Saved tables and checks

The plotting notebook reads the files produced above. Macro-only scores predict 4 or 7 classes; the other model families predict 25 regions. Earlier nested joint ANOVA figures use preserved held-out predictions because their original training source is unavailable.

In [ ]:
cell_type_summary = save_comparison_tables(PROJECT_DIR, DATA_FILE, RESULTS_DIR)
display(cell_type_summary.head(12))
print(f"Validated {len(cell_type_summary)} cell-type / model-family rows")
